In [1]:
import pandas as pd

In [2]:
import timm, torch
# pretrained encoders are hosted on HuggingFace under mwalmsley/
model = timm.create_model("hf_hub:mwalmsley/zoobot-encoder-convnext_nano", pretrained=True).eval()

In [3]:
acts = {}
layer = model.stages[2]                       # choose a layer to visualise
layer.register_forward_hook(lambda m, i, o: acts.update(out=o))

def dream(img, steps=100, lr=0.01, channel=None):
    x = img.clone().requires_grad_(True)      # (1,3,H,W), normalised
    for _ in range(steps):
        model(x)
        a = acts["out"]
        loss = a[:, channel].mean() if channel is not None else a.norm()
        loss.backward()
        with torch.no_grad():
            x += lr * x.grad / (x.grad.abs().mean() + 1e-8)
            x.grad.zero_()
    return x.detach()

In [11]:
df = pd.read_csv("./data_5/dataset.csv")
print(df.head())

                       file_path  label  redshift
0  data_5/lens_png/lens_2000.png      1  2.595677
1  data_5/lens_png/lens_2001.png      1  4.813061
2  data_5/lens_png/lens_2002.png      1  2.762269
3  data_5/lens_png/lens_2003.png      1  2.204104
4  data_5/lens_png/lens_2004.png      1  2.510576


In [ ]:
print(df.info())